It will pick the latest folder inside data/outputs/, then do:
- clean_text.txt
- chunks.json
- embeddings.npy
- faiss.index
- processing_metadata.json

In [4]:
import re
import json
from pathlib import Path

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


OUTPUT_DIR = Path("data/outputs")


def get_latest_uuid_folder(output_dir=OUTPUT_DIR):
    output_dir = Path(output_dir)

    if not output_dir.exists():
        raise FileNotFoundError(f"Output folder not found: {output_dir}")

    folders = [folder for folder in output_dir.iterdir() if folder.is_dir()]

    if not folders:
        raise FileNotFoundError("No UUID folders found in data/outputs")

    latest_folder = max(folders, key=lambda folder: folder.stat().st_mtime)

    return latest_folder


def load_extracted_text(job_folder):
    text_path = job_folder / "all_extracted_text.txt"

    if not text_path.exists():
        raise FileNotFoundError(f"Missing file: {text_path}")

    with open(text_path, "r", encoding="utf-8") as file:
        return file.read()


def clean_text(raw_text):
    text = raw_text

    # remove null characters
    text = text.replace("\x00", " ")

    # normalize spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    # remove too many blank lines
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)

    # fix spaces before punctuation
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    # remove long repeated symbols
    text = re.sub(r"[-_=]{4,}", " ", text)

    clean_lines = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            clean_lines.append("")
            continue

        # skip tiny noisy lines
        if len(line) <= 2:
            continue

        clean_lines.append(line)

    cleaned_text = "\n".join(clean_lines).strip()

    return cleaned_text


def chunk_text(cleaned_text, chunk_size=400, chunk_overlap=80):
    words = cleaned_text.split()

    if not words:
        raise ValueError("No words found after cleaning.")

    chunks = []
    start = 0
    chunk_id = 0

    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk_value = " ".join(chunk_words)

        chunks.append({
            "chunk_id": chunk_id,
            "start_word": start,
            "end_word": min(end, len(words)),
            "text": chunk_value
        })

        chunk_id += 1
        start += chunk_size - chunk_overlap

    return chunks


def generate_embeddings(
    chunks,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    batch_size=8
):
    # load lightweight embedding model
    embedder = SentenceTransformer(model_name)

    texts = [chunk["text"] for chunk in chunks]

    embeddings = embedder.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return np.array(embeddings).astype("float32")


def build_faiss_index(embeddings):
    dimension = embeddings.shape[1]

    # inner product works like cosine similarity because embeddings are normalized
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    return index


def save_processed_outputs(job_folder, cleaned_text, chunks, embeddings, index, metadata):
    clean_text_path = job_folder / "clean_text.txt"
    chunks_path = job_folder / "chunks.json"
    embeddings_path = job_folder / "embeddings.npy"
    faiss_path = job_folder / "faiss.index"
    metadata_path = job_folder / "processing_metadata.json"

    with open(clean_text_path, "w", encoding="utf-8") as file:
        file.write(cleaned_text)

    with open(chunks_path, "w", encoding="utf-8") as file:
        json.dump(chunks, file, indent=2)

    np.save(embeddings_path, embeddings)

    faiss.write_index(index, str(faiss_path))

    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2)

    return {
        "clean_text_path": str(clean_text_path),
        "chunks_path": str(chunks_path),
        "embeddings_path": str(embeddings_path),
        "faiss_index_path": str(faiss_path),
        "metadata_path": str(metadata_path)
    }


def process_latest_output_folder(
    output_dir=OUTPUT_DIR,
    chunk_size=400,
    chunk_overlap=80,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    batch_size=8
):
    latest_folder = get_latest_uuid_folder(output_dir)

    raw_text = load_extracted_text(latest_folder)

    cleaned_text = clean_text(raw_text)

    chunks = chunk_text(
        cleaned_text,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    embeddings = generate_embeddings(
        chunks=chunks,
        model_name=model_name,
        batch_size=batch_size
    )

    index = build_faiss_index(embeddings)

    metadata = {
        "job_folder": str(latest_folder),
        "source_file": "all_extracted_text.txt",
        "cleaned_text_file": "clean_text.txt",
        "num_chunks": len(chunks),
        "embedding_model": model_name,
        "embedding_shape": list(embeddings.shape),
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "batch_size": batch_size,
        "vector_index": "FAISS IndexFlatIP",
        "similarity": "inner product on normalized embeddings"
    }

    saved_files = save_processed_outputs(
        job_folder=latest_folder,
        cleaned_text=cleaned_text,
        chunks=chunks,
        embeddings=embeddings,
        index=index,
        metadata=metadata
    )

    return {
        "latest_folder": str(latest_folder),
        "num_chunks": len(chunks),
        "embedding_shape": embeddings.shape,
        "saved_files": saved_files
    }

In [5]:
result = process_latest_output_folder()

print(json.dumps(result, indent=2, default=str))

Batches: 100%|██████████| 20/20 [00:02<00:00,  9.29it/s]

{
  "latest_folder": "data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4",
  "num_chunks": 156,
  "embedding_shape": [
    156,
    384
  ],
  "saved_files": {
    "clean_text_path": "data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4/clean_text.txt",
    "chunks_path": "data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4/chunks.json",
    "embeddings_path": "data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4/embeddings.npy",
    "faiss_index_path": "data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4/faiss.index",
    "metadata_path": "data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4/processing_metadata.json"
  }
}
